In [33]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
import statsmodels.api as sm

# Load the dataset
df = pd.read_excel("data1-telematics_syn_orignal data.xlsx", index_col=0)

# Data Cleaning and Feature Engineering
# Define age groups
age_dict = {
    'young_drivers': {'age_bracket': '16-25', 'bins': [16, 25]},
    'middle_drivers': {'age_bracket': '26-40', 'bins': [26, 40]},
    'mature_drivers': {'age_bracket': '41-60', 'bins': [41, 60]},
    'senior_drivers': {'age_bracket': '61+', 'bins': [61, df['Insured.age'].max()]}
}
age_bin_list = [16]
age_bin_list.extend([age_dict[x]['bins'][1] for x in age_dict])
age_bin_labels = [age_dict[x]['age_bracket'] for x in age_dict]
df['age_group'] = pd.cut(df['Insured.age'], bins=age_bin_list, labels=age_bin_labels, include_lowest=True)

# Define credit score groups
credit_score_bins = [300, 600, 700, 800, 850]
credit_score_labels = ['Poor', 'Fair', 'Good', 'Excellent']
df['credit_score_group'] = pd.cut(df['Credit.score'], bins=credit_score_bins, labels=credit_score_labels, include_lowest=True)

# Define annual mileage groups
annual_miles_bins = [0, 10000, 15000, 20000, df['Annual.miles.drive'].max()]
annual_miles_labels = ['0-10k', '10k-15k', '15k-20k', '20k+']
df['annual_miles_group'] = pd.cut(df['Annual.miles.drive'], bins=annual_miles_bins, labels=annual_miles_labels, include_lowest=True)

# Define car age groups
car_age_bins = [0, 5, 10, 15, df['Car.age'].max()]
car_age_labels = ['0-5', '6-10', '11-15', '15+']
df['car_age_group'] = pd.cut(df['Car.age'], bins=car_age_bins, labels=car_age_labels, include_lowest=True)

# Define years no claims groups
years_noclaims_bins = [0, 2, 4, 6, df['Years.noclaims'].max()]
years_noclaims_labels = ['0-2', '3-4', '5-6', '6+']
df['years_noclaims_group'] = pd.cut(df['Years.noclaims'], bins=years_noclaims_bins, labels=years_noclaims_labels, include_lowest=True)

# Remove original columns that have been grouped
columns_to_remove = ['Insured.age', 'Credit.score', 'Annual.miles.drive', 'Car.age', 'Years.noclaims']
df.drop(columns=columns_to_remove, inplace=True)

# Handle missing values
numeric_cols = df.select_dtypes(include=['number']).columns
non_numeric_cols = df.select_dtypes(exclude=['number']).columns

# Fill NaN values in numeric columns with the mean
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())

# Fill NaN values in non-numeric columns with the mode
for col in non_numeric_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# Define target variables
df['Severity'] = df.apply(lambda row: row['AMT_Claim'] if row['NB_Claim'] > 0 else 0, axis=1)
df['Loss_Cost'] = df['NB_Claim'] * df['Severity']

# Save prepared data
df.to_csv('_exp4_prepared_data.csv', index=False)


In [34]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

# Function to calculate performance metrics
def calculate_metrics(y_true, y_pred):
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return rmse, mae, r2

# Function to plot group results
def plot_group_results(results_df, group_column, metric, title, ylabel):
    plt.figure(figsize=(14, 8))
    sns.barplot(x=group_column, y=metric, data=results_df)
    plt.xticks(rotation=90)
    plt.title(title)
    plt.ylabel(ylabel)
    plt.show()


In [36]:
# Load prepared data
df = pd.read_csv('_exp4_prepared_data.csv')

# Define cross-validation strategy
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Initialize results dictionary
results_glm = {
    'Group': [],
    'Model': 'GLM',
    'RMSE_Frequency': [],
    'MAE_Frequency': [],
    'R2_Frequency': [],
    'RMSE_Severity': [],
    'MAE_Severity': [],
    'R2_Severity': [],
    'RMSE_Loss_Cost': [],
    'MAE_Loss_Cost': [],
    'R2_Loss_Cost': []
}

# Extract group columns programmatically
group_columns = [col for col in df.columns if any(prefix in col for prefix in ['age_group', 'credit_score_group', 'annual_miles_group', 'car_age_group', 'years_noclaims_group'])]

for group in group_columns:
    unique_groups = df[group].unique()
    for category in unique_groups:
        print(f"Evaluating {group}: {category}")

        # Filter data for the specific category
        group_mask = df[group] == category
        if group_mask.sum() < 5:
            print(f"Not enough samples for {group}: {category}. Skipping this category.")
            continue
        
        # Prepare group-specific data
        X_group = df.loc[group_mask].drop(columns=['NB_Claim', 'Severity', 'Loss_Cost'])
        y_group_freq = df.loc[group_mask, 'NB_Claim']
        y_group_severity = df.loc[group_mask, 'Severity']
        y_group_loss_cost = df.loc[group_mask, 'Loss_Cost']
        
        metrics = {
            'RMSE_Frequency': [],
            'MAE_Frequency': [],
            'R2_Frequency': [],
            'RMSE_Severity': [],
            'MAE_Severity': [],
            'R2_Severity': [],
            'RMSE_Loss_Cost': [],
            'MAE_Loss_Cost': [],
            'R2_Loss_Cost': []
        }

        for train_index, test_index in cv.split(X_group):
            X_train, X_test = X_group.iloc[train_index], X_group.iloc[test_index]
            y_train_freq, y_test_freq = y_group_freq.iloc[train_index], y_group_freq.iloc[test_index]
            y_train_severity, y_test_severity = y_group_severity.iloc[train_index], y_group_severity.iloc[test_index]
            y_train_loss_cost, y_test_loss_cost = y_group_loss_cost.iloc[train_index], y_group_loss_cost.iloc[test_index]
            
            # Add intercept
            X_train_with_intercept = sm.add_constant(X_train)
            X_test_with_intercept = sm.add_constant(X_test)

            # Train and predict claim frequency
            glm_claim_frequency = sm.GLM(y_train_freq, X_train_with_intercept, family=sm.families.Poisson())
            glm_claim_frequency_results = glm_claim_frequency.fit()
            y_glm_pred_claim_frequency = glm_claim_frequency_results.predict(X_test_with_intercept)
            
            # Train and predict severity
            glm_severity = sm.GLM(y_train_severity, X_train_with_intercept, family=sm.families.Gaussian())
            glm_severity_results = glm_severity.fit()
            y_glm_pred_severity = glm_severity_results.predict(X_test_with_intercept)
            
            # Calculate loss cost
            y_glm_pred_loss_cost = y_glm_pred_claim_frequency * y_glm_pred_severity
            
            # Collect metrics
            rmse_freq, mae_freq, r2_freq = calculate_metrics(y_test_freq, y_glm_pred_claim_frequency)
            metrics['RMSE_Frequency'].append(rmse_freq)
            metrics['MAE_Frequency'].append(mae_freq)
            metrics['R2_Frequency'].append(r2_freq)
            
            rmse_severity, mae_severity, r2_severity = calculate_metrics(y_test_severity, y_glm_pred_severity)
            metrics['RMSE_Severity'].append(rmse_severity)
            metrics['MAE_Severity'].append(mae_severity)
            metrics['R2_Severity'].append(r2_severity)
            
            rmse_loss_cost, mae_loss_cost, r2_loss_cost = calculate_metrics(y_test_loss_cost, y_glm_pred_loss_cost)
            metrics['RMSE_Loss_Cost'].append(rmse_loss_cost)
            metrics['MAE_Loss_Cost'].append(mae_loss_cost)
            metrics['R2_Loss_Cost'].append(r2_loss_cost)
        
        # Aggregate metrics for this group and category
        results_glm['Group'].append(f"{group}_{category}")
        results_glm['RMSE_Frequency'].append(np.mean(metrics['RMSE_Frequency']))
        results_glm['MAE_Frequency'].append(np.mean(metrics['MAE_Frequency']))
        results_glm['R2_Frequency'].append(np.mean(metrics['R2_Frequency']))
        results_glm['RMSE_Severity'].append(np.mean(metrics['RMSE_Severity']))
        results_glm['MAE_Severity'].append(np.mean(metrics['MAE_Severity']))
        results_glm['R2_Severity'].append(np.mean(metrics['R2_Severity']))
        results_glm['RMSE_Loss_Cost'].append(np.mean(metrics['RMSE_Loss_Cost']))
        results_glm['MAE_Loss_Cost'].append(np.mean(metrics['MAE_Loss_Cost']))
        results_glm['R2_Loss_Cost'].append(np.mean(metrics['R2_Loss_Cost']))

# Convert results to DataFrame
results_glm_df = pd.DataFrame(results_glm)

# Save results
results_glm_df.to_csv('_exp4_glm_results.csv', index=False)

Evaluating age_group: 41-60


ValueError: Pandas data cast to numpy dtype of object. Check input data with np.asarray(data).